# Autoencoders: De la Teoría a la Práctica con PyTorch

Este cuaderno explora los **autoencoders** (codificadores automáticos), una familia de redes neuronales no supervisadas con amplia variedad de aplicaciones en aprendizaje profundo. Abordaremos su historia, arquitectura, variantes principales y ejemplos prácticos usando PyTorch.

---
## Sección 1: Introducción Teórica

### 1.1 Historia y Contexto

Los autoencoders tienen sus raíces en los trabajos pioneros sobre redes neuronales con retropropagación. En **1986**, Rumelhart, Hinton y Williams demostraron que las redes multicapa entrenadas con retropropagación podían aprender representaciones internas eficientes de los datos *(Rumelhart, Hinton & Williams, 1986)*. La arquitectura básica de autoencoder surgió de estos experimentos como una red que intentaba reconstruir su propia entrada.

Un hito decisivo llegó en **2006** cuando Hinton y Salakhutdinov publicaron su influyente trabajo en *Science*, demostrando que las redes profundas preentrenadas capa a capa (usando Restricted Boltzmann Machines) podían superar a métodos clásicos como PCA en tareas de reducción de dimensionalidad *(Hinton & Salakhutdinov, 2006)*. Este resultado revitalizó el interés en el aprendizaje profundo y en los autoencoders en particular.

Desde entonces, los autoencoders han evolucionado hacia múltiples variantes (sparse, denoising, variational) y se han convertido en una herramienta fundamental del aprendizaje no supervisado y generativo.

### 1.2 Arquitectura General

Un autoencoder consta de tres componentes principales:

```
Entrada (x)  →  [ENCODER]  →  Espacio Latente (z)  →  [DECODER]  →  Reconstrucción (x̂)
```

1. **Encoder (Codificador):** Red neuronal que comprime la entrada `x` de alta dimensión hacia una representación compacta `z` en el espacio latente. Esta compresión fuerza a la red a aprender las características más relevantes de los datos.

2. **Espacio latente (Bottleneck / Código):** Representación interna de dimensión reducida. También llamado *cuello de botella* porque toda la información debe pasar por este espacio comprimido. La dimensión del espacio latente es un hiperparámetro clave: demasiado pequeño impide la reconstrucción; demasiado grande permite memorizar sin aprender estructura.

3. **Decoder (Decodificador):** Red neuronal que reconstruye la entrada original `x̂` a partir del código latente `z`. Su arquitectura es generalmente simétrica a la del encoder.

La siguiente figura ilustra esta arquitectura:

```
        Encoder                  Decoder
   ┌────────────────┐       ┌────────────────┐
   │  784 → 256     │       │  2  → 256      │
x ─┤  256 → 64      ├─ z ──┤  256 → 784     ├─ x̂
   │  64  →  2      │       │                │
   └────────────────┘       └────────────────┘
         (compresión)           (reconstrucción)
```

### 1.3 Descripción Matemática

Sea `x ∈ ℝⁿ` un vector de entrada. El autoencoder aprende dos funciones:

**Encoder:**
$$z = f_{\theta}(x) = \sigma(W_e \cdot x + b_e)$$

donde `z ∈ ℝᵈ` con `d ≪ n` es el código latente, y `σ` es una función de activación (e.g., ReLU).

**Decoder:**
$$\hat{x} = g_{\phi}(z) = \sigma(W_d \cdot z + b_d)$$

donde `x̂ ∈ ℝⁿ` es la reconstrucción de la entrada original.

**Función de pérdida:** El objetivo es minimizar la pérdida de reconstrucción entre `x` y `x̂`:

- **Error Cuadrático Medio (MSE)** — para datos continuos:
$$\mathcal{L}(x, \hat{x}) = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{x}_i)^2$$

- **Entropía Cruzada Binaria (BCE)** — para datos en `[0, 1]` interpretados como probabilidades:
$$\mathcal{L}(x, \hat{x}) = -\frac{1}{n} \sum_{i=1}^{n} \left[ x_i \log(\hat{x}_i) + (1 - x_i) \log(1 - \hat{x}_i) \right]$$

El entrenamiento busca los parámetros `θ` y `φ` que minimicen esta pérdida mediante descenso de gradiente (retropropagación):
$$\theta^*, \phi^* = \arg\min_{\theta, \phi} \mathbb{E}_{x \sim p_{\text{data}}} \left[ \mathcal{L}(x, g_{\phi}(f_{\theta}(x))) \right]$$

### 1.4 Tipos de Autoencoders

| Tipo | Descripción | Caso de uso principal |
|------|-------------|----------------------|
| **Vanilla / Simple** | Arquitectura básica encoder-decoder con pérdida de reconstrucción estándar | Reducción de dimensionalidad, compresión |
| **Sparse** | Añade una penalización de dispersión (L1 o KL) al espacio latente para forzar activaciones escasas | Extracción de características interpretables |
| **Denoising (DAE)** | Se entrena con entradas corrompidas (ruido) para reconstruir la versión limpia | Eliminación de ruido, robustez |
| **Variational (VAE)** | El espacio latente es una distribución probabilística (media y varianza); usa el truco de reparametrización | Generación de nuevos datos, espacio latente continuo |

#### Autoencoder Variacional (VAE) — Nota especial

El VAE *(Kingma & Welling, 2013)* aprende a mapear la entrada a una **distribución** `q(z|x) = N(μ, σ²)` en lugar de un punto. La función de pérdida incluye dos términos:
$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathcal{L}_{\text{reconstrucción}}}_{\text{fidelidad}} + \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{regularización}}$$

El término KL fuerza al espacio latente a aproximarse a una distribución normal estándar `N(0, I)`, permitiendo **generar nuevas muestras** al decodificar puntos del espacio latente.

### 1.5 Aplicaciones Principales

Los autoencoders tienen un amplio espectro de aplicaciones en visión computacional, procesamiento de señales y análisis de datos:

1. **Reducción de dimensionalidad:** Proyectar datos de alta dimensión a espacios compactos para visualización o preprocesamiento. A diferencia de PCA, los autoencoders pueden capturar relaciones **no lineales** en los datos.

2. **Eliminación de ruido (Denoising):** Los denoising autoencoders aprenden a separar la señal del ruido, lo que los hace útiles en imágenes médicas, señales de audio y datos de sensores.

3. **Detección de anomalías:** Al entrenarse solo con datos normales, el autoencoder tiene un error de reconstrucción bajo para muestras normales y alto para anomalías (ejemplos atípicos), lo cual permite detectarlas.

4. **Generación de datos:** Los VAEs pueden generar muestras nuevas y plausibles interpolando en el espacio latente continuo.

5. **Aprendizaje de representaciones:** El espacio latente captura características semánticas útiles para tareas downstream (clasificación, clustering).

6. **Compresión de información:** En dominio de imágenes o audio, se pueden usar como compresores lossy aprendidos.

### 1.6 Diferencias entre Autoencoders y PCA

| Característica | PCA | Autoencoder |
|----------------|-----|-------------|
| **Tipo de transformación** | Lineal | No lineal (con funciones de activación no lineales) |
| **Solución** | Analítica (descomposición espectral) | Iterativa (descenso de gradiente) |
| **Capacidad expresiva** | Limitada a subespacios lineales | Alta — puede capturar manifolds no lineales |
| **Interpretabilidad** | Alta (componentes principales ortogonales) | Menor (representación distribuida) |
| **Escalabilidad** | Costosa para datos de muy alta dimensión | Escalable con mini-batches |
| **Flexibilidad** | Rígida (una sola formulación) | Alta (sparse, denoising, variational, etc.) |
| **Garantía de óptimo global** | Sí (solución exacta) | No (puede converger a mínimos locales) |

> **Nota:** Si el autoencoder usa exclusivamente activaciones lineales y pérdida MSE, la solución aprendida es equivalente a PCA *(Baldi & Hornik, 1989)*. La no linealidad es lo que le otorga mayor poder expresivo.

---
## Sección 2: Implementación Base con PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Semillas para reproducibilidad
torch.manual_seed(1)
np.random.seed(1)

# Dispositivo: GPU si está disponible, si no CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

### 2.1 Definición del Autoencoder Simple (Vanilla)

Implementamos un autoencoder completamente conectado (fully-connected) con:
- **Encoder:** `input_dim → hidden_dim → latent_dim`
- **Decoder:** `latent_dim → hidden_dim → input_dim`

Se usa `ReLU` en las capas intermedias y `Sigmoid` en la salida del decoder para mantener los valores reconstruidos en `[0, 1]`.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=2):
        super(Autoencoder, self).__init__()
        # Encoder: comprime la entrada al espacio latente
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        # Decoder: reconstruye la entrada desde el espacio latente
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()  # Salida en [0,1] para datos normalizados
        )

    def forward(self, x):
        z = self.encoder(x)       # Codificación
        x_hat = self.decoder(z)   # Decodificación (reconstrucción)
        return x_hat

    def encode(self, x):
        """Devuelve únicamente la representación latente z."""
        return self.encoder(x)

# Ejemplo rápido para verificar las dimensiones
modelo_prueba = Autoencoder(input_dim=784, hidden_dim=256, latent_dim=2)
x_prueba = torch.randn(8, 784)  # Lote de 8 imágenes aplanadas
salida_prueba = modelo_prueba(x_prueba)
codigo_prueba = modelo_prueba.encode(x_prueba)
print(f'Forma de entrada:         {x_prueba.shape}')
print(f'Forma del código latente: {codigo_prueba.shape}')
print(f'Forma de reconstrucción:  {salida_prueba.shape}')

---
## Sección 3: Ejemplo 1 — Autoencoder Simple con MNIST

MNIST es un dataset canónico de dígitos escritos a mano (0-9), con 60 000 imágenes de entrenamiento y 10 000 de prueba, cada una de tamaño 28×28 píxeles. Lo usaremos para:

1. Entrenar el autoencoder vanilla.
2. Visualizar originales vs. reconstrucciones.
3. Analizar la curva de pérdida.
4. Explorar el espacio latente 2D coloreado por clase.

In [ ]:
# ── Carga del dataset MNIST ────────────────────────────────────────────────
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False)

print(f'Imágenes de entrenamiento: {len(train_dataset)}')
print(f'Imágenes de prueba:        {len(test_dataset)}')
print(f'Tamaño de cada imagen:     {train_dataset[0][0].shape}')

In [ ]:
# ── Crear el autoencoder y configurar el entrenamiento ─────────────────────
autoencoder = Autoencoder(input_dim=784, hidden_dim=256, latent_dim=2).to(device)
criterion   = nn.MSELoss()                             # Pérdida de reconstrucción MSE
optimizer   = optim.Adam(autoencoder.parameters(), lr=1e-3)

num_epochs  = 10
train_losses = []

print(f'Parámetros totales del modelo: {sum(p.numel() for p in autoencoder.parameters()):,}')

# ── Bucle de entrenamiento ─────────────────────────────────────────────────
for epoch in range(num_epochs):
    autoencoder.train()
    epoch_loss = 0.0
    for batch_idx, (data, _) in enumerate(train_loader):
        # Aplanar imágenes 28x28 → vector de 784 dimensiones
        data = data.view(data.size(0), -1).to(device)

        optimizer.zero_grad()         # Limpiar gradientes
        output = autoencoder(data)    # Paso hacia adelante
        loss   = criterion(output, data)  # Calcular pérdida
        loss.backward()               # Retropropagación
        optimizer.step()              # Actualizar pesos

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Época [{epoch+1:02d}/{num_epochs}]  |  Pérdida promedio: {avg_loss:.6f}')

In [ ]:
# ── Visualización: imágenes originales vs. reconstruidas ──────────────────
autoencoder.eval()
with torch.no_grad():
    # Tomar un lote del conjunto de prueba
    test_data, _ = next(iter(test_loader))
    test_data_flat = test_data.view(test_data.size(0), -1).to(device)
    reconstructed  = autoencoder(test_data_flat).cpu()

n_imagenes = 10
fig, axes = plt.subplots(2, n_imagenes, figsize=(15, 3))

for i in range(n_imagenes):
    # Fila superior: imágenes originales
    axes[0, i].imshow(test_data[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')

    # Fila inferior: reconstrucciones
    axes[1, i].imshow(reconstructed[i].view(28, 28), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, rotation=0, labelpad=55)
axes[1, 0].set_ylabel('Reconstruida', fontsize=11, rotation=0, labelpad=55)

fig.suptitle('Autoencoder Simple — Originales vs. Reconstrucciones (MNIST)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Curva de pérdida de entrenamiento ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, num_epochs + 1), train_losses, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Pérdida MSE', fontsize=12)
ax.set_title('Curva de Pérdida — Autoencoder Simple (MNIST)', fontsize=13)
ax.grid(True, alpha=0.4)
ax.set_xticks(range(1, num_epochs + 1))
plt.tight_layout()
plt.show()

print(f'Pérdida inicial: {train_losses[0]:.6f}')
print(f'Pérdida final:   {train_losses[-1]:.6f}')
print(f'Reducción:       {(1 - train_losses[-1]/train_losses[0]) * 100:.1f}%')

In [ ]:
# ── Visualización del espacio latente 2D ──────────────────────────────────
autoencoder.eval()
codigos_lista  = []
etiquetas_lista = []

with torch.no_grad():
    for data, etiquetas in test_loader:
        data_flat = data.view(data.size(0), -1).to(device)
        z = autoencoder.encode(data_flat).cpu().numpy()
        codigos_lista.append(z)
        etiquetas_lista.append(etiquetas.numpy())

codigos   = np.concatenate(codigos_lista,   axis=0)  # (10000, 2)
etiquetas = np.concatenate(etiquetas_lista, axis=0)  # (10000,)

# Paleta de 10 colores para los dígitos 0-9
colores = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(9, 7))
for digito in range(10):
    mascara = etiquetas == digito
    ax.scatter(
        codigos[mascara, 0], codigos[mascara, 1],
        c=[colores[digito]], label=str(digito),
        s=5, alpha=0.5
    )

ax.set_xlabel('Dimensión latente 1', fontsize=12)
ax.set_ylabel('Dimensión latente 2', fontsize=12)
ax.set_title('Espacio Latente 2D — Autoencoder Simple (MNIST)', fontsize=13)
ax.legend(title='Dígito', markerscale=3, fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Sección 4: Ejemplo 2 — Denoising Autoencoder

Un **Denoising Autoencoder (DAE)** se entrena recibiendo versiones corrompidas de la entrada y aprendiendo a devolver la versión limpia. El modelo aprende a descartar el ruido y conservar las características estructurales de los datos.

**Proceso de entrenamiento:**
1. Tomar imagen original `x`.
2. Agregar ruido gaussiano: `x̃ = x + ε`, donde `ε ~ N(0, σ²)`.
3. Pasar `x̃` por el autoencoder para obtener `x̂`.
4. Minimizar `L(x, x̂)` — comparando la reconstrucción con la imagen **original limpia**.

Usamos un espacio latente más grande (`latent_dim=64`) que en el vanilla, ya que necesitamos preservar más información para realizar la tarea de denoising.

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=64):
        super(DenoisingAutoencoder, self).__init__()
        # Encoder con dos capas ocultas para mayor capacidad
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()
        )
        # Decoder simétrico
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def add_noise(data, noise_factor=0.3):
    """Añade ruido gaussiano a los datos y recorta al rango [0, 1]."""
    noisy = data + noise_factor * torch.randn_like(data)
    return torch.clamp(noisy, 0., 1.)


print('Clase DenoisingAutoencoder definida correctamente.')

In [ ]:
# ── Entrenamiento del Denoising Autoencoder ───────────────────────────────
dae       = DenoisingAutoencoder(input_dim=784, hidden_dim=256, latent_dim=64).to(device)
criterion = nn.MSELoss()
optimizer_dae = optim.Adam(dae.parameters(), lr=1e-3)

num_epochs_dae = 10
train_losses_dae = []

for epoch in range(num_epochs_dae):
    dae.train()
    epoch_loss = 0.0
    for data, _ in train_loader:
        data_flat  = data.view(data.size(0), -1).to(device)  # Imagen limpia (objetivo)
        noisy_data = add_noise(data_flat, noise_factor=0.3)   # Imagen ruidosa (entrada)

        optimizer_dae.zero_grad()
        output = dae(noisy_data)                # Reconstruir desde la entrada ruidosa
        loss   = criterion(output, data_flat)   # Comparar con la imagen limpia
        loss.backward()
        optimizer_dae.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses_dae.append(avg_loss)
    print(f'Época [{epoch+1:02d}/{num_epochs_dae}]  |  Pérdida DAE: {avg_loss:.6f}')

In [ ]:
# ── Visualización: ruidosa → reconstruida → original ──────────────────────
dae.eval()
with torch.no_grad():
    test_data_dae, _ = next(iter(test_loader))
    test_flat   = test_data_dae.view(test_data_dae.size(0), -1).to(device)
    noisy_test  = add_noise(test_flat, noise_factor=0.3)
    denoised    = dae(noisy_test).cpu()
    noisy_cpu   = noisy_test.cpu()

n_imagenes = 10
fig, axes = plt.subplots(3, n_imagenes, figsize=(15, 4.5))

for i in range(n_imagenes):
    # Fila 1: imagen original
    axes[0, i].imshow(test_data_dae[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    # Fila 2: imagen con ruido
    axes[1, i].imshow(noisy_cpu[i].view(28, 28), cmap='gray')
    axes[1, i].axis('off')
    # Fila 3: imagen reconstruida (denoised)
    axes[2, i].imshow(denoised[i].view(28, 28), cmap='gray')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original',   fontsize=10, rotation=0, labelpad=55)
axes[1, 0].set_ylabel('Con ruido',  fontsize=10, rotation=0, labelpad=55)
axes[2, 0].set_ylabel('Reconstruida', fontsize=10, rotation=0, labelpad=55)

fig.suptitle('Denoising Autoencoder — Original, Ruidosa y Reconstruida (MNIST)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparación de curvas de pérdida: Vanilla vs DAE ─────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, num_epochs + 1), train_losses,
        marker='o', label='Autoencoder Simple', color='steelblue', linewidth=2)
ax.plot(range(1, num_epochs_dae + 1), train_losses_dae,
        marker='s', label='Denoising Autoencoder', color='darkorange', linewidth=2)
ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Pérdida MSE', fontsize=12)
ax.set_title('Comparación de Pérdidas — Vanilla vs. Denoising Autoencoder', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
ax.set_xticks(range(1, num_epochs + 1))
plt.tight_layout()
plt.show()

---
## Sección 5: Ejemplo 3 — Reducción de Dimensionalidad: Autoencoder vs. PCA

Comparamos la capacidad de reducción de dimensionalidad del autoencoder (2D) entrenado anteriormente con **PCA** aplicado sobre las mismas imágenes. Ambos métodos proyectan cada imagen de 784 dimensiones a un espacio 2D; la diferencia clave es que PCA solo puede capturar variaciones **lineales**, mientras que el autoencoder puede modelar estructuras **no lineales**.

Se colorean los puntos por clase (dígito 0-9) para evaluar visualmente la separabilidad de las representaciones.

In [ ]:
# ── Extraer datos del conjunto de prueba para la comparación ──────────────
imagenes_np   = []
etiquetas_np  = []

autoencoder.eval()
codigos_ae = []

with torch.no_grad():
    for data, etiquetas in test_loader:
        data_flat = data.view(data.size(0), -1)
        imagenes_np.append(data_flat.numpy())
        etiquetas_np.append(etiquetas.numpy())
        z = autoencoder.encode(data_flat.to(device)).cpu().numpy()
        codigos_ae.append(z)

imagenes_np  = np.concatenate(imagenes_np,  axis=0)  # (10000, 784)
etiquetas_np = np.concatenate(etiquetas_np, axis=0)  # (10000,)
codigos_ae   = np.concatenate(codigos_ae,   axis=0)  # (10000, 2)

# ── Aplicar PCA a los mismos datos ────────────────────────────────────────
escalador = StandardScaler()
imagenes_escaladas = escalador.fit_transform(imagenes_np)

pca = PCA(n_components=2, random_state=1)
codigos_pca = pca.fit_transform(imagenes_escaladas)

var_explicada = pca.explained_variance_ratio_.sum() * 100
print(f'Varianza explicada por 2 componentes PCA: {var_explicada:.2f}%')

In [ ]:
# ── Visualización comparativa: Autoencoder vs. PCA ────────────────────────
colores = plt.cm.tab10(np.linspace(0, 1, 10))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for digito in range(10):
    mascara = etiquetas_np == digito

    # Panel izquierdo: Autoencoder
    ax1.scatter(
        codigos_ae[mascara, 0], codigos_ae[mascara, 1],
        c=[colores[digito]], label=str(digito), s=5, alpha=0.5
    )
    # Panel derecho: PCA
    ax2.scatter(
        codigos_pca[mascara, 0], codigos_pca[mascara, 1],
        c=[colores[digito]], label=str(digito), s=5, alpha=0.5
    )

ax1.set_title('Autoencoder — Espacio Latente 2D', fontsize=13)
ax1.set_xlabel('Dimensión 1', fontsize=11)
ax1.set_ylabel('Dimensión 2', fontsize=11)
ax1.legend(title='Dígito', markerscale=3, fontsize=9, loc='best')
ax1.grid(True, alpha=0.3)

ax2.set_title(f'PCA — 2 Componentes ({var_explicada:.1f}% varianza)', fontsize=13)
ax2.set_xlabel('PC 1', fontsize=11)
ax2.set_ylabel('PC 2', fontsize=11)
ax2.legend(title='Dígito', markerscale=3, fontsize=9, loc='best')
ax2.grid(True, alpha=0.3)

fig.suptitle('Reducción de Dimensionalidad 2D: Autoencoder vs. PCA (MNIST)', fontsize=14)
plt.tight_layout()
plt.show()

**Observaciones:**
- El **autoencoder** puede aprender fronteras no lineales entre clases, generando clústeres más compactos y separados en el espacio latente.
- **PCA** captura la mayor varianza lineal posible con solo 2 componentes, pero los clústeres pueden solaparse más al no poder modelar no linealidades.
- La calidad de la separación en ambos casos depende del número de épocas, la arquitectura y la complejidad del dataset.

---
## Sección 6: Ejemplo 4 — Detección de Anomalías

Una aplicación poderosa de los autoencoders es la **detección de anomalías**. La idea es:

1. Entrenar el autoencoder **únicamente** con datos de una clase (datos normales).
2. El modelo aprende a reconstruir bien esa clase específica.
3. Al presentarle datos de una clase diferente (anomalía), el error de reconstrucción será **significativamente mayor**.

En este ejemplo, entrenamos el autoencoder solo con el dígito **"1"** y luego evaluamos el error de reconstrucción tanto para dígitos **"1"** (normal) como para dígitos **"7"** (anomalía), que comparten cierta similitud visual pero tienen estructuras distintas.

In [ ]:
# ── Filtrar el dataset: solo dígito "1" para entrenamiento ────────────────
indices_1_train = [i for i, (_, y) in enumerate(train_dataset) if y == 1]
indices_7_train = [i for i, (_, y) in enumerate(train_dataset) if y == 7]

indices_1_test  = [i for i, (_, y) in enumerate(test_dataset) if y == 1]
indices_7_test  = [i for i, (_, y) in enumerate(test_dataset) if y == 7]

dataset_1_train = torch.utils.data.Subset(train_dataset, indices_1_train)
dataset_1_test  = torch.utils.data.Subset(test_dataset,  indices_1_test)
dataset_7_test  = torch.utils.data.Subset(test_dataset,  indices_7_test)

loader_1_train = DataLoader(dataset_1_train, batch_size=128, shuffle=True)
loader_1_test  = DataLoader(dataset_1_test,  batch_size=256, shuffle=False)
loader_7_test  = DataLoader(dataset_7_test,  batch_size=256, shuffle=False)

print(f'Muestras de entrenamiento (dígito 1): {len(dataset_1_train)}')
print(f'Muestras de prueba (dígito 1):        {len(dataset_1_test)}')
print(f'Muestras de prueba (dígito 7):        {len(dataset_7_test)}')

In [ ]:
# ── Definir y entrenar el autoencoder de anomalías ────────────────────────
# Usamos un espacio latente un poco más grande (latent_dim=8) para capturar
# suficiente información del dígito '1' sin sobreajustar.
ae_anomaly   = Autoencoder(input_dim=784, hidden_dim=256, latent_dim=8).to(device)
criterion_an = nn.MSELoss(reduction='none')  # Error por píxel (sin reducir)
optimizer_an = optim.Adam(ae_anomaly.parameters(), lr=1e-3)

num_epochs_an = 15
losses_an = []

for epoch in range(num_epochs_an):
    ae_anomaly.train()
    epoch_loss = 0.0
    for data, _ in loader_1_train:
        data_flat = data.view(data.size(0), -1).to(device)
        optimizer_an.zero_grad()
        output = ae_anomaly(data_flat)
        loss   = criterion_an(output, data_flat).mean()  # Reducir manualmente
        loss.backward()
        optimizer_an.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader_1_train)
    losses_an.append(avg_loss)
    print(f'Época [{epoch+1:02d}/{num_epochs_an}]  |  Pérdida (dígito 1): {avg_loss:.6f}')

In [ ]:
# ── Calcular error de reconstrucción por muestra ──────────────────────────
def calcular_errores(modelo, loader, dispositivo):
    """Retorna el MSE promedio por imagen para todo el loader."""
    modelo.eval()
    errores = []
    with torch.no_grad():
        for data, _ in loader:
            data_flat  = data.view(data.size(0), -1).to(dispositivo)
            output     = modelo(data_flat)
            # Error cuadrático medio por imagen
            err_por_img = ((output - data_flat) ** 2).mean(dim=1).cpu().numpy()
            errores.append(err_por_img)
    return np.concatenate(errores)


errores_1 = calcular_errores(ae_anomaly, loader_1_test, device)  # Normal
errores_7 = calcular_errores(ae_anomaly, loader_7_test, device)  # Anomalía

print(f'Error medio dígito 1 (normal):   {errores_1.mean():.6f} ± {errores_1.std():.6f}')
print(f'Error medio dígito 7 (anomalía): {errores_7.mean():.6f} ± {errores_7.std():.6f}')
print(f'Razón (anomalía / normal):        {errores_7.mean() / errores_1.mean():.2f}x')

In [ ]:
# ── Visualización 1: Distribución de errores ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(errores_1, bins=50, alpha=0.7, color='steelblue',
        label=f'Dígito 1 (normal)   — media: {errores_1.mean():.4f}', density=True)
ax.hist(errores_7, bins=50, alpha=0.7, color='tomato',
        label=f'Dígito 7 (anomalía) — media: {errores_7.mean():.4f}', density=True)

# Umbral de detección: media + 2 desviaciones estándar del dígito normal
umbral = errores_1.mean() + 2 * errores_1.std()
ax.axvline(umbral, color='black', linestyle='--', linewidth=2,
           label=f'Umbral de detección: {umbral:.4f}')

ax.set_xlabel('Error de reconstrucción (MSE)', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.set_title('Detección de Anomalías — Error de Reconstrucción\n'
             'Autoencoder entrenado solo con dígito "1"', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualización 2: Imágenes originales vs. reconstruidas (1 y 7) ────────
ae_anomaly.eval()

# Obtener un lote de cada clase
datos_1, _ = next(iter(loader_1_test))
datos_7, _ = next(iter(loader_7_test))

with torch.no_grad():
    recon_1 = ae_anomaly(datos_1.view(datos_1.size(0), -1).to(device)).cpu()
    recon_7 = ae_anomaly(datos_7.view(datos_7.size(0), -1).to(device)).cpu()

n_viz = 8
fig, axes = plt.subplots(4, n_viz, figsize=(15, 6))

for i in range(n_viz):
    # Fila 1: original dígito 1
    axes[0, i].imshow(datos_1[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    # Fila 2: reconstrucción dígito 1
    axes[1, i].imshow(recon_1[i].view(28, 28), cmap='gray')
    axes[1, i].axis('off')
    # Fila 3: original dígito 7
    axes[2, i].imshow(datos_7[i].squeeze(), cmap='gray')
    axes[2, i].axis('off')
    # Fila 4: reconstrucción dígito 7 (borrosa por ser anomalía)
    axes[3, i].imshow(recon_7[i].view(28, 28), cmap='gray')
    axes[3, i].axis('off')

axes[0, 0].set_ylabel('"1" original',   fontsize=9, rotation=0, labelpad=65)
axes[1, 0].set_ylabel('"1" reconstruido', fontsize=9, rotation=0, labelpad=65)
axes[2, 0].set_ylabel('"7" original',   fontsize=9, rotation=0, labelpad=65)
axes[3, 0].set_ylabel('"7" reconstruido', fontsize=9, rotation=0, labelpad=65)

fig.suptitle('Detección de Anomalías — Reconstrucción de dígito "1" (normal) vs "7" (anomalía)',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Métricas de detección con el umbral definido ──────────────────────────
# Un error > umbral se clasifica como anomalía
umbral = errores_1.mean() + 2 * errores_1.std()

vp = (errores_7 > umbral).sum()   # Verdaderos positivos (7 detectado como anomalía)
fn = (errores_7 <= umbral).sum()  # Falsos negativos (7 no detectado)
vn = (errores_1 <= umbral).sum()  # Verdaderos negativos (1 clasificado como normal)
fp = (errores_1 > umbral).sum()   # Falsos positivos (1 clasificado como anomalía)

precision_anomalia = vp / (vp + fp) if (vp + fp) > 0 else 0
recall_anomalia    = vp / (vp + fn) if (vp + fn) > 0 else 0

print('=== Métricas de Detección de Anomalías ===')
print(f'Umbral de detección:  {umbral:.6f}')
print(f'Verdaderos positivos ("7" → anomalía): {vp}')
print(f'Falsos negativos    ("7" → normal):    {fn}')
print(f'Verdaderos negativos ("1" → normal):   {vn}')
print(f'Falsos positivos     ("1" → anomalía): {fp}')
print(f'Precisión:  {precision_anomalia:.3f}')
print(f'Recall:     {recall_anomalia:.3f}')

---
## Sección 7: Referencias Bibliográficas

Las siguientes referencias fundamentan el contenido teórico y práctico desarrollado en este cuaderno:

1. **Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986).** Learning representations by back-propagating errors. *Nature*, 323(6088), 533–536. https://doi.org/10.1038/323533a0

2. **Hinton, G. E., & Salakhutdinov, R. R. (2006).** Reducing the dimensionality of data with neural networks. *Science*, 313(5786), 504–507. https://doi.org/10.1126/science.1127647

3. **Baldi, P., & Hornik, K. (1989).** Neural networks and principal component analysis: Learning from examples without local minima. *Neural Networks*, 2(1), 53–58. https://doi.org/10.1016/0893-6080(89)90014-2

4. **Vincent, P., Larochelle, H., Bengio, Y., & Manzagol, P. A. (2008).** Extracting and composing robust features with denoising autoencoders. In *Proceedings of the 25th International Conference on Machine Learning (ICML)*, 1096–1103. https://doi.org/10.1145/1390156.1390294

5. **Kingma, D. P., & Welling, M. (2013).** Auto-encoding variational bayes. *arXiv preprint arXiv:1312.6114*. https://arxiv.org/abs/1312.6114

6. **Goodfellow, I., Bengio, Y., & Courville, A. (2016).** *Deep Learning* (Capítulo 14: Autoencoders). MIT Press. https://www.deeplearningbook.org/

7. **LeCun, Y., Cortes, C., & Burges, C. J. C. (1998).** The MNIST database of handwritten digits. *AT&T Labs*. http://yann.lecun.com/exdb/mnist/

8. **Paszke, A., Gross, S., Massa, F., et al. (2019).** PyTorch: An imperative style, high-performance deep learning library. In *Advances in Neural Information Processing Systems (NeurIPS)*, 32. https://papers.neurips.cc/paper/2019/hash/bdbca288fee7f92f2bfa9f7012727740-Abstract.html